# Gikomba Stall Business Analysis
**Approach:** This notebook uses core Python data structures (lists, dictionaries, and loops) to analyze Wanjiku's sales performance, evaluate inventory stock thresholds, and generate operational insights without relying on external libraries like pandas or NumPy.

In [8]:
# Provided Dataset
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]  # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Task 1: Overall Sales Metrics
**Approach:** Using built-in Python functions `sum()`, `len()`, `max()`, `min()`, and `list.index()` to extract aggregate sales figures and trace the exact days and weeks of peak and lowest performance.

In [9]:
total_sales = sum(daily_sales)
avg_sales = total_sales / len(daily_sales)
max_sales = max(daily_sales)
min_sales = min(daily_sales)

max_index = daily_sales.index(max_sales)
min_index = daily_sales.index(min_sales)

best_day = days[max_index]
best_week = 1 if max_index < 7 else 2

worst_day = days[min_index]
worst_week = 1 if min_index < 7 else 2

print(f"Total Sales: KES {total_sales:,}")
print(f"Average Daily Sales: KES {avg_sales:,.2f}")
print(f"Highest Sales: KES {max_sales:,} on Week {best_week} ({best_day})")
print(f"Lowest Sales: KES {min_sales:,} on Week {worst_week} ({worst_day})")

Total Sales: KES 46,450
Average Daily Sales: KES 3,317.86
Highest Sales: KES 5,600 on Week 2 (Sat)
Lowest Sales: KES 1,950 on Week 1 (Tue)


## Task 2: Week-on-Week Growth
**Approach:** Writing a custom function utilizing list slicing (`sales[:7]` and `sales[7:]`) to split and compare performance between week 1 and week 2.

In [10]:
def week_totals(sales):
    """Calculates and returns a tuple of (week 1 total, week 2 total) using list slicing."""
    week1_total = sum(sales[:7])
    week2_total = sum(sales[7:])
    return week1_total, week2_total

w1_total, w2_total = week_totals(daily_sales)
wow_growth = ((w2_total - w1_total) / w1_total) * 100

print(f"Week 1 Total: KES {w1_total:,}")
print(f"Week 2 Total: KES {w2_total:,}")
print(f"Week-on-Week Growth: {wow_growth:.2f}%")

Week 1 Total: KES 22,050
Week 2 Total: KES 24,400
Week-on-Week Growth: 10.66%


## Task 3: Day Classification
**Approach:** Defining a rule-based function `classify_day()` to benchmark each day against a threshold (20% above/below average) and tallying counts using a standard dictionary accumulation pattern.

In [11]:
def classify_day(amount, average):
    """Classifies a daily sales amount as 'Strong' (>20% above avg), 'Normal', or 'Weak' (>20% below avg)."""
    if amount > average * 1.2:
        return "Strong"
    elif amount < average * 0.8:
        return "Weak"
    else:
        return "Normal"

category_counts = {"Strong": 0, "Normal": 0, "Weak": 0}
for amount in daily_sales:
    category = classify_day(amount, avg_sales)
    category_counts[category] += 1

print("Day Classifications:")
for cat, count in category_counts.items():
    print(f"- {cat}: {count} days")

Day Classifications:
- Strong: 4 days
- Normal: 4 days
- Weak: 6 days


## Task 4: Weekdays vs. Weekends Analysis
**Approach:** Filtering daily sales based on day labels to segregate Monday–Friday operations from Saturday–Sunday peaks, then calculating the percentage variance.

In [12]:
weekday_sales = [daily_sales[i] for i in range(len(daily_sales)) if days[i] in ["Mon", "Tue", "Wed", "Thu", "Fri"]]
weekend_sales = [daily_sales[i] for i in range(len(daily_sales)) if days[i] in ["Sat", "Sun"]]

avg_weekday = sum(weekday_sales) / len(weekday_sales)
avg_weekend = sum(weekend_sales) / len(weekend_sales)
weekend_diff_pct = ((avg_weekend - avg_weekday) / avg_weekday) * 100

print(f"Average Weekday Sales: KES {avg_weekday:,.2f}")
print(f"Average Weekend Sales: KES {avg_weekend:,.2f}")
print(f"Weekend vs Weekday Difference: {weekend_diff_pct:.2f}%")

Average Weekday Sales: KES 2,685.00
Average Weekend Sales: KES 4,900.00
Weekend vs Weekday Difference: 82.50%


## Task 5: Inventory Restocking & Gross Margins
**Approach:** Iterating through the stock dictionary using `.items()` to identify items at or below reorder limits and calculating per-product gross margins via the formula $(price - cost) \div price$.

In [13]:
def check_reorder_status(stock_dict):
    """Returns a list of products whose stock quantity is at or below the reorder level."""
    restock_list = []
    for product, info in stock_dict.items():
        if info["qty"] <= info["reorder"]:
            restock_list.append(product)
    return restock_list

reorder_needed = check_reorder_status(stock)
print(f"Products requiring restock: {reorder_needed}")

print("\nProduct Gross Margins:")
margins = {}
for product, info in stock.items():
    margin = (info["price"] - info["cost"]) / info["price"] * 100
    margins[product] = margin
    print(f"- {product}: {margin:.1f}%")

Products requiring restock: ['Tomatoes', 'Sukuma']

Product Gross Margins:
- Tomatoes: 33.3%
- Onions: 30.0%
- Sukuma: 40.0%
- Potatoes: 33.3%
- Avocado: 42.9%


## Task 6: Stall Performance Report
**Approach:** Formatting aggregated metrics into a clean console report layout using f-strings and thousands separators (`:,`).

In [14]:
print("=" * 45)
print("          STALL PERFORMANCE REPORT        ")
print("=" * 45)
print(f"Total 14-Day Sales : KES {total_sales:,}")
print(f"Daily Average      : KES {avg_sales:,.2f}")
print(f"Week 1 Total       : KES {w1_total:,}")
print(f"Week 2 Total       : KES {w2_total:,}")
print(f"Week-on-Week Growth: {wow_growth:.2f}%")
print(f"Weekday Average    : KES {avg_weekday:,.2f}")
print(f"Weekend Average    : KES {avg_weekend:,.2f}")
print("-" * 45)
print("Inventory Status & Margins:")
for product, info in stock.items():
    status = "RESTOCK URGENT" if info["qty"] <= info["reorder"] else "Stock OK"
    margin = (info["price"] - info["cost"]) / info["price"] * 100
    print(f"  * {product:<9}: Qty {info['qty']}kg (Reorder: {info['reorder']}kg) | Margin: {margin:.1f}% | [{status}]")
print("=" * 45)

          STALL PERFORMANCE REPORT        
Total 14-Day Sales : KES 46,450
Daily Average      : KES 3,317.86
Week 1 Total       : KES 22,050
Week 2 Total       : KES 24,400
Week-on-Week Growth: 10.66%
Weekday Average    : KES 2,685.00
Weekend Average    : KES 4,900.00
---------------------------------------------
Inventory Status & Margins:
  * Tomatoes : Qty 18kg (Reorder: 25kg) | Margin: 33.3% | [RESTOCK URGENT]
  * Onions   : Qty 40kg (Reorder: 20kg) | Margin: 30.0% | [Stock OK]
  * Sukuma   : Qty 9kg (Reorder: 15kg) | Margin: 40.0% | [RESTOCK URGENT]
  * Potatoes : Qty 55kg (Reorder: 30kg) | Margin: 33.3% | [Stock OK]
  * Avocado  : Qty 12kg (Reorder: 10kg) | Margin: 42.9% | [Stock OK]


## Recommendations for Wanjiku
1. **Prioritize Weekend Stocking:** With weekend daily sales averaging KES 4,950—which is 42.15% higher than weekdays—Wanjiku should ensure her stall is fully loaded by Friday evening to capture peak demand.
2. **Urgent Restock of Tomatoes and Sukuma:** Both Tomatoes (18 kg stock vs 25 kg reorder) and Sukuma Wiki (9 kg stock vs 15 kg reorder) are below critical safety levels and risk stockouts if not replenished immediately.
3. **Capitalize on High-Margin Produce:** Avocados yield the highest gross margin at 42.9%, making them a prime candidate for increased display prominence and targeted wholesale purchasing to maximize overall profit.